# 5. Classify posts (batch)

For each resolved post from notebook 3, classify it against its paper as an **endorsement**, **flag**, or **irrelevant** using Claude.

Inputs: `output/<RUN_ID>/resolved_posts.parquet` (notebook 3), `output/<RUN_ID>/papers.parquet` (notebook 1), `output/<RUN_ID>/posts.parquet` (notebook 2, for the `post_id → ro_id` map).
Outputs: `output/<RUN_ID>/classified.parquet`.

**Batch is the default**; toggle `USE_BATCH = False` for the single-call fallback during demos. Batch is half-price and avoids rate limits, but it's asynchronous — the cell polls until done.

See `PIPELINE_PLAN.md` § S22.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.resolve()
load_dotenv(REPO_ROOT / ".env")
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted([p.name for p in OUTPUT_ROOT.iterdir() if (p / "resolved_posts.parquet").exists()])
    if not candidates:
        raise FileNotFoundError("Run notebook 3 first.")
    RUN_ID = candidates[-1]
OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}")

Using run id: 2026-06-28T21-34


In [2]:
import pandas as pd
from altendor.enrich.text_resolver import ResolvedPost
from altendor.classify.classifier import PaperCtx

resolved_df = pd.read_parquet(OUT_DIR / "resolved_posts.parquet")
papers_df = pd.read_parquet(OUT_DIR / "papers.parquet").set_index("ro_id")

# Also need to know each post's paper. Bring in posts.parquet for the post_id → ro_id mapping.
posts_df = pd.read_parquet(OUT_DIR / "posts.parquet")
post_to_ro = dict(zip(posts_df["post_id"], posts_df["ro_id"]))
print(f"{len(resolved_df)} resolved posts × {len(papers_df)} papers")

17218 resolved posts × 10 papers


## Build classification specs

Each spec pairs a resolved post with its paper's `PaperCtx` (title + abstract). Posts whose `ro_id` isn't in `papers_df` are skipped.

In [3]:
from altendor.classify.batch import BatchRequestSpec

def _str(value) -> str:
    """Coerce a pandas cell to a clean string (NaN/None → '')."""
    if value is None:
        return ""
    if isinstance(value, float) and value != value:  # NaN
        return ""
    return str(value)


def _str_or_none(value) -> str | None:
    if value is None:
        return None
    if isinstance(value, float) and value != value:
        return None
    s = str(value)
    return s if s else None


def _row_to_resolved(row) -> ResolvedPost:
    return ResolvedPost(
        post_id=_str(row["post_id"]),
        platform=_str(row["platform"]),
        text=_str(row["text"]),
        author_handle=_str_or_none(row.get("author_handle")),
        author_id=_str_or_none(row.get("author_id")),
        url=_str(row["url"]),
        created_at=_str(row["created_at"]),
        raw_title=_str(row["raw_title"]),
        text_confidence=_str(row["text_confidence"]) or "high",
    )


def _ro_to_paper(ro_id: str) -> PaperCtx | None:
    if ro_id not in papers_df.index:
        return None
    p = papers_df.loc[ro_id]
    title = _str(p["title"])
    if not title:
        return None
    return PaperCtx(title=title, abstract=_str_or_none(p.get("abstract")))


# resolved_df may have duplicate post_ids when a post mentions several papers
# (notebook 3 iterates the exploded posts_df from notebook 2). Dedupe by the
# (ro_id, post_id) custom_id so each unique pair is classified exactly once.
specs_by_custom_id: dict[str, BatchRequestSpec] = {}
skipped_no_paper = 0
skipped_dupe = 0
skipped_no_text = 0
for _, row in resolved_df.iterrows():
    ro_id = post_to_ro.get(row["post_id"])
    if ro_id is None:
        skipped_no_paper += 1
        continue
    paper = _ro_to_paper(ro_id)
    if paper is None:
        skipped_no_paper += 1
        continue
    custom_id = f"{ro_id}__{row['post_id']}"
    if custom_id in specs_by_custom_id:
        skipped_dupe += 1
        continue
    post = _row_to_resolved(row)
    if not post.text:
        # Nothing meaningful to classify.
        skipped_no_text += 1
        continue
    specs_by_custom_id[custom_id] = BatchRequestSpec(
        custom_id=custom_id,
        post=post,
        paper=paper,
    )

specs: list[BatchRequestSpec] = list(specs_by_custom_id.values())
print(
    f"Built {len(specs)} specs; "
    f"skipped {skipped_no_paper} (no matching paper), "
    f"{skipped_dupe} (duplicate post_id × paper), "
    f"{skipped_no_text} (empty post text)"
)

Built 17186 specs; skipped 0 (no matching paper), 32 (duplicate post_id × paper), 0 (empty post text)


## Subsample per paper (cost / time control)

Top sci-of-sci papers can attract thousands of Altmetric mentions, which blows past the budget if classified in full. Cap to `N_PER_PAPER` posts per paper so coverage stays balanced across the 10 papers. Set `N_PER_PAPER = None` to disable.

In [4]:
import random

N_PER_PAPER: int | None = 100   # set to None to classify every spec
RANDOM_SEED = 20260629

if N_PER_PAPER is not None:
    rng = random.Random(RANDOM_SEED)
    by_paper: dict[str, list] = {}
    for s in specs:
        ro_id = s.custom_id.split("__", 1)[0]
        by_paper.setdefault(ro_id, []).append(s)
    sampled: list = []
    for ro_id, group in by_paper.items():
        rng.shuffle(group)
        sampled.extend(group[:N_PER_PAPER])
    print(
        f"Sampled down to {len(sampled)} specs "
        f"({N_PER_PAPER} per paper × {len(by_paper)} papers; was {len(specs)})."
    )
    specs = sampled

Sampled down to 1000 specs (100 per paper × 10 papers; was 17186).


## Configure run mode

`USE_BATCH=True` submits to the Anthropic Batch API (half-price, async). `False` falls back to per-post `classify_post` calls — useful for demos / debugging on a handful of items. `MAX_TO_CLASSIFY` caps the spec list for dev runs.

In [5]:
import anthropic

USE_BATCH = True
MAX_TO_CLASSIFY: int | None = None  # cap during dev runs; None = process all
POLL_INTERVAL_SEC = 30

if MAX_TO_CLASSIFY is not None:
    specs = specs[:MAX_TO_CLASSIFY]
    print(f"Capped at {len(specs)} specs for this run.")

client = anthropic.Anthropic()

## Submit + parse

Batch path: `submit_batch` returns a `batch_id`; `poll_until_done` blocks until the batch finishes, printing per-tick status; `parse_batch_results` reads the JSONL output.

Single-call path: a plain loop over `classify_post`, capturing exceptions into `error_reason`.

In [ ]:
from altendor.classify.batch import submit_batch, poll_until_done, parse_batch_results, BatchResultRow
from altendor.classify.classifier import classify_post

results: list[BatchResultRow]
if USE_BATCH:
    batch_id = submit_batch(client, specs)
    print(f"Submitted batch: {batch_id}")
    def _tick(batch):
        print(f"... status={batch.processing_status}  counts={batch.request_counts}")
    poll_until_done(client, batch_id, interval_s=POLL_INTERVAL_SEC, on_tick=_tick)
    results = parse_batch_results(client, batch_id)
else:
    results = []
    for s in specs:
        try:
            cr = classify_post(client, s.post, s.paper)
            results.append(BatchResultRow(custom_id=s.custom_id, result=cr, error_reason=None))
        except Exception as exc:
            results.append(BatchResultRow(custom_id=s.custom_id, result=None, error_reason=str(exc)))
    print(f"Single-call mode: {len(results)} done.")

Submitted batch: msgbatch_01MBckzmqYCYckjwJFxLT9Jj
... status=in_progress  counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=1000, succeeded=0)


## Flatten results to a DataFrame

One row per result. `kind` is `endorsement` / `flag` / `irrelevant` / `None` (on error). Per-kind fields are populated only on the matching branch; others stay `None`.

In [ ]:
from altendor.classify.schema import Endorsement, Flag, Irrelevant

rows = []
for r in results:
    ro_id, post_id = r.custom_id.split("__", 1)
    doi = papers_df.loc[ro_id, "doi"] if ro_id in papers_df.index else None
    base = {"post_id": post_id, "ro_id": ro_id, "doi": doi,
            "kind": None, "claim_text": None, "magnitude_dB": None,
            "criterion": None, "reasoning": None,
            "category": None, "rationale": None, "reason": None,
            "error_reason": r.error_reason}
    if isinstance(r.result, Endorsement):
        base.update(kind="endorsement", claim_text=r.result.claim_text,
                    magnitude_dB=r.result.magnitude_dB, criterion=r.result.criterion,
                    reasoning=r.result.reasoning)
    elif isinstance(r.result, Flag):
        base.update(kind="flag", category=r.result.category, rationale=r.result.rationale)
    elif isinstance(r.result, Irrelevant):
        base.update(kind="irrelevant", reason=r.result.reason)
    rows.append(base)

classified_df = pd.DataFrame(rows)
print(classified_df["kind"].value_counts(dropna=False).to_string())

## Write outputs

Persist `classified.parquet` and append `stage_5_classify_batch` to the run's `manifest.json`.

In [ ]:
import json

out_path = OUT_DIR / "classified.parquet"
classified_df.to_parquet(out_path, index=False)

manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_5_classify_batch"] = {
    "mode": "batch" if USE_BATCH else "single",
    "n_input": int(len(specs)),
    "n_output": int(len(classified_df)),
    "by_kind": {str(k): int(v) for k, v in classified_df["kind"].value_counts(dropna=False).items()},
    "n_errors": int(classified_df["error_reason"].notna().sum()),
}
manifest.setdefault("output_files", []).append(str(out_path.relative_to(REPO_ROOT)))
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Wrote {out_path}")